In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import re
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.dates as mdates
import pickle
import seaborn as sns
sns.set(style="whitegrid")


import platform
system_name = platform.system()
if system_name == 'Windows':
    plt.rcParams['font.sans-serif'] = ['SimHei']  # Windows 用黑体
elif system_name == 'Darwin':
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS'] # Mac 用 Arial Unicode

plt.rcParams['axes.unicode_minus'] = False # 解决负号显示问题

In [2]:
data = pd.read_csv(r"E:\26Spring\Final\code\panel.csv")

In [3]:
HSCI_PATH = r"E:\26Spring\Final\data\HSCI.csv"
ANN_PATH  = r"E:\26Spring\Final\data\hk_announcement.csv"

# ====== 读数 + 清洗 ======
members = (
    pd.read_csv(HSCI_PATH)
      .assign(
          date=lambda d: pd.to_datetime(d["date"], errors="coerce"),
          sid=lambda d: d["sid"].astype(str).str.strip(),
      )
)
ann = (
    pd.read_csv(ANN_PATH, encoding="gbk")
      .assign(
          info_date=lambda d: pd.to_datetime(d["info_date"], errors="coerce"),
          order_book_id=lambda d: d["order_book_id"].astype(str).str.strip(),
          title=lambda d: d["title"].astype(str),
      )
)

print("members:", members.shape, members.columns.tolist())
print("ann:", ann.shape, ann.columns.tolist())

members = members.dropna(subset=["date", "sid"]).copy()
ann = ann.dropna(subset=["info_date", "order_book_id", "title"]).copy()

# ====== sid -> order_book_id ======
members["order_book_id"] = (
    members["sid"].astype(str)
    .str.replace(".HK", "", regex=False)
    .str.zfill(5)
    .radd("")
    .add(".XHKG")
)

print(members[["date", "sid", "order_book_id"]].head())

# ====== 同日 inner join ======
ann_hksc = ann.merge(
    members[["date", "order_book_id"]],
    left_on=["info_date", "order_book_id"],
    right_on=["date", "order_book_id"],
    how="inner",
)

print("ann_hksc:", ann_hksc.shape)
print(ann_hksc[["info_date", "order_book_id", "title"]].head())

# ====== 汇总表 1 ======
date_range_str = lambda s: f"{s.min().date()} ~ {s.max().date()}"

table1 = pd.DataFrame([
    {
        "数据集": "港股通名单 (HSCI)",
        "时间范围": date_range_str(members["date"]),
        "股票数量(去重 order_book_id)": members["order_book_id"].nunique(),
        "记录条数": len(members),
        "备注": "date×sid",
    },
    {
        "数据集": "港股公告 (Announcement)",
        "时间范围": date_range_str(ann["info_date"]),
        "股票数量(去重 order_book_id)": ann["order_book_id"].nunique(),
        "记录条数": len(ann),
        "备注": "全量公告",
    },
    {
        "数据集": "港股通×公告 (Join 后样本)",
        "时间范围": date_range_str(ann_hksc["info_date"]),
        "股票数量(去重 order_book_id)": ann_hksc["order_book_id"].nunique(),
        "记录条数": len(ann_hksc),
        "备注": "inner join on (info_date, order_book_id)",
    },
])
table1

# ====== 汇总表 2 ======
table2 = pd.DataFrame([
    {"筛选步骤": "原始公告数据", "剩余公告数": len(ann), "剩余股票数": ann["order_book_id"].nunique()},
    {"筛选步骤": "匹配港股通当日成分股(同日 inner join)", "剩余公告数": len(ann_hksc), "剩余股票数": ann_hksc["order_book_id"].nunique()},
])
table2

# ====== 样本期截取 ======
start, end = pd.Timestamp("2015-01-01"), pd.Timestamp("2025-09-30")
work = ann_hksc[ann_hksc["info_date"].between(start, end)].copy()
print("work:", work.shape, work["info_date"].min(), work["info_date"].max())


members: (2019624, 2) ['date', 'sid']
ann: (1305817, 11) ['order_book_id', 'info_date', 'media', 'title', 'language', 'file_type', 'announcement_link', 'first_category', 'second_category', 'third_category', 'rice_create_tm']
        date      sid order_book_id
0 2005-01-03  0001.HK    00001.XHKG
1 2005-01-03  0715.HK    00715.XHKG
2 2005-01-03  0716.HK    00716.XHKG
3 2005-01-03  0728.HK    00728.XHKG
4 2005-01-03  0737.HK    00737.XHKG
ann_hksc: (790869, 12)
   info_date order_book_id                                              title
0 2014-08-04    00001.XHKG          长江实业(00001)公告及通告 - [內幕消息]內幕消息(158KB, PDF)
1 2014-08-04    00001.XHKG  长江实业(00001)月報表截至二零一四年七月三十一日止月份之股份發行人的證券變動月報表(9...
2 2014-08-07    00001.XHKG  长江实业(00001)公告及通告 - [其他-業務發展最新情況]自願披露 - 有關合營公司提...
3 2014-08-14    00001.XHKG  长江实业(00001)通函 - [其他]致非登記持有人之通知信函 - 有關二零一四年度中期報...
4 2014-08-14    00001.XHKG      长江实业(00001)通函 - [其他]非登記持有人適用之申請表格(183KB, PDF)
work: (779099, 12) 2015-01-02 00:00:00 2025-09-30 00:00:00


In [4]:
# ====== 标题关键词筛回购 ======
kw = [
    "回购", "回購", "股份回购", "股份回購", "购回", "購回",
    "股份購回", "股份购回",
    "repurchase", "buyback", "share repurchase", "share buyback",
]
pattern = "|".join(kw)
mask = work["title"].astype(str).str.contains(pattern, case=False, na=False)
rep = work[mask].copy()

print("rep:", rep.shape)
print(rep[["info_date", "order_book_id", "title"]].head(10))

# ---------- Step 1: 相同链接去重（仅对非空链接生效） ----------
link_col_candidates = ["announcement_link"]  
link_col = next((c for c in link_col_candidates if c in rep.columns), None)

if link_col is None:
    print("rep 中没有找到链接列（候选：announcement_link/link/url/pdf_link），跳过 Step 1 链接去重。")
    rep1 = rep.copy()
else:
    rep1 = rep.copy()
    nonnull = rep1[link_col].notna() & rep1[link_col].astype(str).str.strip().ne("")
    rep1_nonnull = rep1.loc[nonnull].drop_duplicates(subset=[link_col], keep="first")
    rep1_null = rep1.loc[~nonnull]  # 保留所有无链接的（不参与链接去重）
    rep1 = pd.concat([rep1_nonnull, rep1_null], axis=0).sort_index()
    print(f"Step1 done: link_col={link_col}, rep -> rep1:", rep.shape, "->", rep1.shape)

# ---------- Step 2: 同日同股，中英双语去重 ----------
# 语言判别：title 里出现中文字符就算中文，否则算英文/其他
rep1 = rep1.assign(
    is_zh = rep1["title"].astype(str).str.contains(r"[\u4e00-\u9fff]", na=False),
    title_len = rep1["title"].astype(str).str.len(),
)

# 对每个 (info_date, order_book_id)：
# - 如果同时存在中文和英文：优先保留中文（is_zh=True），若同语言多条则保留更长标题
# - 如果只存在一种语言：保留该组的“最优一条”（同样按 is_zh/title_len 排序等价于保留更长）
rep1_sorted = rep1.sort_values(
    by=["info_date", "order_book_id", "is_zh", "title_len"],
    ascending=[True, True, False, False]
)

rep2 = rep1_sorted.drop_duplicates(subset=["info_date", "order_book_id"], keep="first").copy()

print("Step2 done: rep1 -> rep2:", rep1.shape, "->", rep2.shape)

rep: (80887, 12)
     info_date order_book_id  \
103 2015-05-21    00001.XHKG   
104 2015-05-21    00001.XHKG   
105 2015-05-21    00001.XHKG   
158 2016-04-12    00001.XHKG   
159 2016-04-12    00001.XHKG   
160 2016-04-12    00001.XHKG   
161 2016-04-12    00001.XHKG   
184 2016-11-17    00001.XHKG   
185 2016-11-18    00001.XHKG   
208 2017-04-05    00001.XHKG   

                                                 title  
103  長和(00001)通函 - [在股東批准的情況下重選或委任董事 / 一般性授權 / 回購股份...  
104  長和(00001)通函 - [在股東批准的情況下重選或委任董事 / 一般性授權 / 回購股份...  
105  長和(00001)通函 - [在股東批准的情況下重選或委任董事 / 一般性授權 / 回購股份...  
158  長和(00001)通函 - [在股東批准的情況下重選或委任董事 / 一般性授權 / 回購股份...  
159  長和(00001)通函 - [在股東批准的情況下重選或委任董事 / 一般性授權 / 回購股份...  
160  長和(00001)通函 - [在股東批准的情況下重選或委任董事 / 一般性授權 / 回購股份...  
161  長和(00001)通函 - [在股東批准的情況下重選或委任董事 / 一般性授權 / 回購股份...  
184         長和(00001)翌日披露報表 - [股份購回]翌日披露報表(179KB, PDF)  
185         長和(00001)翌日披露報表 - [股份購回]翌日披露報表(180KB, PDF)  
208  長和(00001)通函 - [在股東批准的情況下重選或委任董事 / 一般性授權 / 回購股份...  
Ste

In [5]:
# =========================
# 1) nextday
# =========================
kw_nextday = [
    "翌日", "次日",
    "next day", "the next day",
    "翌日披露报表", "翌日披露報表",
    "next day disclosure return"
]
pat_nextday = "|".join(kw_nextday)

# # =========================
# # 2) execution：实施 / 已执行回购
# # =========================
# kw_execution = [
#     "股份購回", "股份购回",
#     "購回股份", "购回股份",
#     "回購股份", "回购股份",
#     "從市場購回股份", "从市场购回股份",
#     "自願公告從市場購回股份", "自愿公告从市场购回股份",
#     "關於實施H股回購的公告", "关于实施H股回购的公告",
#     "實施H股回購情況", "实施H股回购情况",
#     "股份回購情況", "股份回购情况",
#     "share repurchase", "share buy-back", "share buyback",
#     "repurchase of shares", "buy-back of shares", "buyback of shares"
# ]
# pat_execution = "|".join(kw_execution)

# =========================
# 2) execution：实施 / 已执行 / 披露回购 / 有意进行回购
# =========================
kw_execution = [
    # 最核心中文（简繁都补）
    "股份購回", "股份购回",
    "股份回購", "股份回购",
    "購回股份", "购回股份",
    "回購股份", "回购股份",

    # 市场买回
    "從市場購回股份", "从市场购回股份",
    "於市場上購回股份", "于市场上购回股份",
    "於市場上回購股份", "于市场上回购股份",
    "在市場上購回股份", "在市场上购回股份",
    "在市場上回購股份", "在市场上回购股份",

    # 自愿/自願公告常见写法
    "自願公告-股份回購", "自願公告 - 股份回購",
    "自愿公告-股份回购", "自愿公告 - 股份回购",
    "自願性公告-股份回購", "自願性公告 - 股份回購",
    "自愿性公告-股份回购", "自愿性公告 - 股份回购",

    "自願公告-股份購回", "自願公告 - 股份購回",
    "自愿公告-股份购回", "自愿公告 - 股份购回",
    "自願性公告-股份購回", "自願性公告 - 股份購回",
    "自愿性公告-股份购回", "自愿性公告 - 股份购回",

    # 执行/实施
    "實施H股回購", "实施H股回购",
    "實施股份回購", "实施股份回购",
    "股份回購情況", "股份回购情况",
    "股份購回情況", "股份购回情况",
    "回購情況", "回购情况",
    "購回情況", "购回情况",

    # 意向 / 启动（建议也纳入 execution）
    "有意於市場上進行股份回購", "有意于市场上进行股份回购",
    "有意於市場上進行股份購回", "有意于市场上进行股份购回",
    "有意進行股份回購", "有意进行股份回购",
    "有意進行股份購回", "有意进行股份购回",

    # 英文
    "share repurchase", "share buy-back", "share buyback",
    "repurchase of shares", "buy-back of shares", "buyback of shares",
    "repurchase of h shares", "repurchase of h-shares"
]

kw_execution += [
    "首次实施回购", "首次實施回購",
    "实施回购", "實施回購",
    "实施情况", "實施情況",
    "进展公告", "進展公告",
    "回购进展", "回購進展",
    "回购部分A股股份", "回購部分A股股份",
    "回购部分H股股份", "回購部分H股股份",
    "回购部分社会公众股份", "回購部分社會公眾股份",
    "股份回购实施情况", "股份回購實施情況",
    "股份回购进展", "股份回購進展"
]

pat_execution = "|".join(kw_execution)

# =========================
# 3) completion：完成
# =========================
kw_completion = [
    "完成", "完畢",
    "completed", "completion",
    "完成回购", "完成回購",
    "回购完成", "回購完成",
    "completion of share repurchase",
    "completed share repurchase"
]
pat_completion = "|".join(kw_completion)

# =========================
# 4) mandate：授权 / 提案 / 方案 / 董事会 / 独立意见
# =========================
kw_mandate = [
    "一般性授权", "一般性授權",
    "回购授权", "回購授權",
    "股份回购授权", "股份回購授權",
    "股份購回授權", "股份购回授权",
    "回购股份一般授权", "回購股份一般授權",
    "回购股份的说明函件", "回購股份的說明函件",
    "購回股份的說明函件", "购回股份的说明函件",
    "购回股份及发行", "購回股份及發行",
    "董事会决议", "董事會決議",
    "独立董事", "獨立董事",
    "回购方案", "回購方案",
    "初步方案",
    "建议授予", "建議授予",
    "general mandate", "repurchase mandate", "buyback mandate",
    "explanatory statement", "repurchase resolution"
]

kw_mandate += [
    "回购预案", "回購預案",
    "回购计划", "回購計劃",
    "回购安排", "回購安排",
    "股份回购方案", "股份回購方案",
    "股份回购计划", "股份回購計劃"
]

kw_mandate += [
    "报告书", "報告書",
    "独立财务顾问报告", "獨立財務顧問報告",
    "独立财务顾问", "獨立財務顧問",
    "前十名股东持股信息", "前十名股東持股信息",
    "回购事项", "回購事項",
    "建议购回H股的一般授权", "建議購回H股的一般授權",
    "建议回购H股的一般授权", "建議回購H股的一般授權"
]

kw_mandate += [
    "授予董事会回购本公司H股之一般授权",
    "授予董事會回購本公司H股之一般授權",
    "回购本公司H股之一般授权",
    "回購本公司H股之一般授權"
]

pat_mandate = "|".join(kw_mandate)


# =========================
# 5) corporate action：要约 / 守则 / 清洗豁免 / 债权人 / 文件流程
# =========================
kw_corporate_action = [
    "公司股份回购守则", "公司股份回購守則",
    "根据《公司股份回购守则》发出的公告", "根據《公司股份回購守則》發出的公告",
    "有条件现金要约", "有條件現金要約",
    "现金要约", "現金要約",
    "清洗豁免",
    "恢复买卖", "恢復買賣",
    "债权人通知", "債權人通知",
    "通知债权人", "通知債權人",
    "要约文件", "要約文件",
    "延迟寄发要约文件", "延遲寄發要約文件",
    "offer document", "whitewash waiver", "takeovers code"
]

kw_corporate_action += [
    "债权人之公告", "債權人之公告",
    "债权人之第二次公告", "債權人之第二次公告",
    "债权人之第三次公告", "債權人之第三次公告"
]

pat_corporate_action = "|".join(kw_corporate_action)

# =========================
# 6) cancellation：注销已回购股份
# =========================
kw_cancellation = [
    "注销已回购", "註銷已回購",
    "注销回购股份", "註銷回購股份",
    "已回购股份注销", "已回購股份註銷",
    "cancellation of repurchased shares",
    "cancelled repurchased shares",
    "cancellation of treasury shares"
]
pat_cancellation = "|".join(kw_cancellation)

# =========================
# 7) irrelevant：伪回购 / 非股票回购
# =========================
kw_irrelevant = [
    "债券质押式回购", "債券質押式回購",
    "质押式回购", "質押式回購",
    "逆回购", "逆回購",
    "repo",
    "限制性股票激励", "限制性股票激勵",
    "限制性股票",
    "股权激励", "股權激勵",
    "回购注销部分限制性股票", "回購註銷部分限制性股票"
]

kw_irrelevant += [
    "未償還的債券", "未偿还的债券",
    "債券之收購要約", "债券之收购要约",
    "債券的收購要約", "债券的收购要约",
    "收購要約之結果", "收购要约之结果",
    "收購要約的結算", "收购要约的结算",
    "有担保债券", "有擔保債券",
    "徵求同意", "征求同意",
    "回購及註銷部份債券", "回购及注销部份债券",
    "回购协议", "回購協議",
    "購回有關位於", "购回有关位于",
    "住宅單位的經濟利益", "住宅单位的经济利益",
    "經濟利益", "经济利益"
]

kw_irrelevant += [
    # ===== 债券 / 可转债 / 永续证券 / 票据 =====
    "债券", "債券",
    "可换股债券", "可換股債券",
    "可转股票据", "可轉股票據",
    "可赎回", "可贖回",
    "永久次级可换股证券", "永久次級可換股證券",
    "可换股证券", "可換股證券",
    "票据", "票據",
    "优先票据", "優先票據",
    "中期票据", "中期票據",
    "债务证券", "債務證券",
    "担保债券", "擔保債券",
    "担保优先票据", "擔保優先票據",

    # ===== 债券 buyback / tender offer / settlement =====
    "回购并注销部分债券", "回購並註銷部分債券",
    "部分购回", "部份購回", "部分購回",
    "購回票據", "购回票据",
    "購回可換股債券", "购回可换股债券",
    "購回可轉股票據", "购回可转股票据",
    "收购要约", "收購要約",
    "回购邀请", "回購邀請",
    "要约邀请", "要約邀請",
    "最终分配通知", "最終分配通知",
    "结算通知", "結算通知",
    "接纳金额", "接納金額",
    "购回价", "購回價",
    "征求同意", "徵求同意",

    # ===== redemption / delisting / notes =====
    "赎回", "贖回",
    "提早赎回", "提早贖回",
    "除牌",
    "撤销上市", "撤銷上市",

    # ===== repo / bond repo =====
    "债券质押式正回购", "債券質押式正回購",
    "正回购交易", "正回購交易",
    "质押式正回购", "質押式正回購",

    # ===== 其他典型债务工具关键词 =====
    "发行人", "發行人",
    "要约人",
    "美元永久次级",
    "零票息",
    "中期票据计划", "中期票據計劃",
    "股份代号：", "股份代號：",
]
pat_irrelevant = "|".join(kw_irrelevant)

pat_irrelevant = "|".join(kw_irrelevant)

In [6]:
rep2 = rep2.assign(
    hit_nextday = rep2["title"].astype(str).str.contains(pat_nextday, case=False, na=False),
    hit_execution = rep2["title"].astype(str).str.contains(pat_execution, case=False, na=False),
    hit_completion = rep2["title"].astype(str).str.contains(pat_completion, case=False, na=False),
    hit_mandate = rep2["title"].astype(str).str.contains(pat_mandate, case=False, na=False),
    hit_corporate_action = rep2["title"].astype(str).str.contains(pat_corporate_action, case=False, na=False),
    hit_cancellation = rep2["title"].astype(str).str.contains(pat_cancellation, case=False, na=False),
    hit_irrelevant_repurchase = rep2["title"].astype(str).str.contains(pat_irrelevant, case=False, na=False),
)

In [7]:
pd.set_option("display.max_colwidth", None)   # title 不截断
pd.set_option("display.width", 200)           # 整体显示更宽
pd.set_option("display.max_columns", None)    # 不省略列

In [8]:
# 七个标签都没命中的记录
rep2_all_false = rep2.loc[
    (~rep2["hit_nextday"]) &
    (~rep2["hit_execution"]) &
    (~rep2["hit_completion"]) &
    (~rep2["hit_mandate"]) &
    (~rep2["hit_corporate_action"]) &
    (~rep2["hit_cancellation"]) &
    (~rep2["hit_irrelevant_repurchase"])
].copy()

print("七个标签全 False 的记录数：", rep2_all_false.shape[0])

display(
    rep2_all_false[
        [
            "info_date", "order_book_id", "title",
            "hit_nextday", "hit_execution", "hit_completion",
            "hit_mandate", "hit_corporate_action",
            "hit_cancellation", "hit_irrelevant_repurchase"
        ]
    ].head(20)
)

七个标签全 False 的记录数： 263


,info_date,order_book_id,title,hit_nextday,hit_execution,hit_completion,hit_mandate,hit_corporate_action,hit_cancellation,hit_irrelevant_repurchase
523924,2016-11-10,02196.XHKG,"复星医药(02196)公告及通告 - [海外监管公告-其他]海外监管公告 – 独立非执行董事关于回购注销部分未解锁限制性A股股票的独立意见(383KB, PDF)",False,False,False,False,False,False,False
315292,2016-11-17,01157.XHKG,"中联重科(01157)公告及通告 - [海外监管公告-证券发行及相关事宜]关于中联重科股份有限公司回购部分A股社会公众股份的法律意见书(392KB, PDF)",False,False,False,False,False,False,False
95238,2016-12-06,00323.XHKG,"马鞍山钢铁股份(00323)公告及通告 - [海外监管公告-其他]关于控股股东进行股票质押回购交易的公告(148KB, PDF)",False,False,False,False,False,False,False
95239,2016-12-13,00323.XHKG,"马鞍山钢铁股份(00323)公告及通告 - [海外监管公告-其他]关于控股股东进行股票质押回购交易进展的公告(354KB, PDF)",False,False,False,False,False,False,False
658251,2017-05-09,03383.XHKG,"雅居乐集团(03383)公告及通告 - [主要交易 / 关连交易]主要及关连交易建议购回冠金30%的股本权益(417KB, PDF)",False,False,False,False,False,False,False
315367,2017-05-24,01157.XHKG,"中联重科(01157)公告及通告 - [海外监管公告-证券发行及相关事宜]关于回购部份A股股份首次实施(547KB, PDF)",False,False,False,False,False,False,False
650538,2017-06-28,03333.XHKG,"中國恒大(03333)公告及通告 - [主要交易 / 條款上的更改]有關主要交易之進一步公告對投資協議的回購義務之修訂(209KB, PDF)",False,False,False,False,False,False,False
220124,2017-07-03,00832.XHKG,"建業地產(00832)公告及通告 - [須予披露的交易]須予披露交易—合作框架協議(1)出售項目公司之股權；及(2)承諾回購項目公司股權(459KB, PDF)",False,False,False,False,False,False,False
304185,2017-11-13,01112.XHKG,"H&H国际控股(01112)公告及通告 - [内幕消息]由Swisse自PGT购回分销权及中国业务购买权(274KB,PDF)",False,False,False,False,False,False,False
163503,2017-12-29,00639.XHKG,"首钢资源(00639)公告及通告 - [关连交易]获豁免关连交易、股份转让补充协议及购回补充协议(216KB, PDF)",False,False,False,False,False,False,False


In [9]:
print("hit_nextday              :", rep2["hit_nextday"].sum())
print("hit_execution            :", rep2["hit_execution"].sum())
print("hit_completion           :", rep2["hit_completion"].sum())
print("hit_mandate              :", rep2["hit_mandate"].sum())
print("hit_corporate_action     :", rep2["hit_corporate_action"].sum())
print("hit_cancellation         :", rep2["hit_cancellation"].sum())
print("hit_irrelevant_repurchase:", rep2["hit_irrelevant_repurchase"].sum())

hit_nextday              : 26222
hit_execution            : 31640
hit_completion           : 222
hit_mandate              : 4483
hit_corporate_action     : 338
hit_cancellation         : 40
hit_irrelevant_repurchase: 1917


In [10]:
# =========================================================
# 统一港股代码格式：
# 00001.XHKG / 1.HK / 0001.hk / 00001.hk  -> 0001.hk
# =========================================================
def normalize_hk_ticker(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip().lower()
    m = re.search(r"(\d+)", s)
    if not m:
        return pd.NA
    num = int(m.group(1))
    return f"{num:04d}.hk"

# =========================================================
# 0. 基础格式统一
# =========================================================
data = data.copy()
rep2 = rep2.copy()

data["date"] = pd.to_datetime(data["date"], errors="coerce").dt.normalize()
rep2["info_date"] = pd.to_datetime(rep2["info_date"], errors="coerce").dt.normalize()

data["ticker_norm"] = data["ticker"].map(normalize_hk_ticker)
rep2["ticker_norm"] = rep2["order_book_id"].map(normalize_hk_ticker)

# 去掉关键键缺失
data = data.dropna(subset=["date", "ticker_norm"]).copy()
rep2 = rep2.dropna(subset=["info_date", "ticker_norm"]).copy()

# =========================================================
# 1. 检查 7 个布尔标签是否存在
# =========================================================
label_cols = [
    "hit_nextday",
    "hit_execution",
    "hit_completion",
    "hit_mandate",
    "hit_corporate_action",
    "hit_cancellation",
    "hit_irrelevant_repurchase"
]

missing_cols = [c for c in label_cols if c not in rep2.columns]
print("missing label cols:", missing_cols)
if len(missing_cols) > 0:
    raise ValueError(f"rep2 缺少这些标签列: {missing_cols}")

# =========================================================
# 2. 7 个布尔标签 -> 1 个唯一 event_type
# =========================================================
priority = [
    "hit_execution",
    "hit_completion",
    "hit_cancellation",
    "hit_mandate",
    "hit_nextday",
    "hit_corporate_action",
    "hit_irrelevant_repurchase"
]

label_to_event = {
    "hit_execution": "execution",
    "hit_completion": "completion",
    "hit_cancellation": "cancellation",
    "hit_mandate": "mandate",
    "hit_nextday": "nextday",
    "hit_corporate_action": "corporate_action",
    "hit_irrelevant_repurchase": "irrelevant_repurchase"
}

event_rank_map = {
    "execution": 1,
    "completion": 2,
    "cancellation": 3,
    "mandate": 4,
    "nextday": 5,
    "corporate_action": 6,
    "irrelevant_repurchase": 7,
    "none": 99
}

rep2[label_cols] = rep2[label_cols].fillna(False).astype(bool)

rep2["event_type"] = pd.Series([pd.NA] * len(rep2), index=rep2.index, dtype="object")
for col in priority:
    mask = rep2["event_type"].isna() & rep2[col]
    rep2.loc[mask, "event_type"] = label_to_event[col]

rep2["event_type"] = rep2["event_type"].fillna("none")
rep2["event_rank"] = rep2["event_type"].map(event_rank_map).astype(int)
rep2["has_event"] = rep2["event_type"].ne("none").astype(bool)
rep2["n_hits"] = rep2[label_cols].sum(axis=1).astype(int)

# =========================================================
# 3. 聚合到 ticker × date
#    同一股票同一天多条公告时，保留优先级最高的一条
# =========================================================
daily_event = (
    rep2.sort_values(["ticker_norm", "info_date", "event_rank", "n_hits"], ascending=[True, True, True, False])
        .drop_duplicates(subset=["ticker_norm", "info_date"], keep="first")
        [["ticker_norm", "info_date", "event_type", "event_rank", "has_event"]]
        .rename(columns={"info_date": "date"})
        .copy()
)

daily_count = (
    rep2[rep2["event_type"] != "none"]
        .groupby(["ticker_norm", "info_date"], as_index=False)
        .size()
        .rename(columns={"info_date": "date", "size": "n_announcements"})
)

daily_event = daily_event.merge(
    daily_count,
    on=["ticker_norm", "date"],
    how="left"
)

daily_event["n_announcements"] = daily_event["n_announcements"].fillna(0).astype(int)

# =========================================================
# 4. merge 回 data
# =========================================================
# 先删掉旧列，避免重复 merge
cols_to_drop = ["event_type", "event_rank", "has_event", "n_announcements"]
data = data.drop(columns=[c for c in cols_to_drop if c in data.columns], errors="ignore")

data = data.merge(
    daily_event,
    on=["ticker_norm", "date"],
    how="left",
    validate="many_to_one"
)

data["event_type"] = data["event_type"].fillna("none")
data["event_rank"] = data["event_rank"].fillna(99).astype(int)
data["has_event"] = data["has_event"].fillna(False).astype(bool)
data["n_announcements"] = data["n_announcements"].fillna(0).astype(int)

# 如需保留原 ticker，同时也可把标准化结果写回
# data["ticker"] = data["ticker_norm"]



missing label cols: []


C:\Users\SHEN\AppData\Local\Temp\ipykernel_21284\1491433364.py:138: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data["has_event"] = data["has_event"].fillna(False).astype(bool)


In [11]:
data

,date,ticker,AdjClose,close,open,high,low,volume,amount,turnover,dy,buyback_amount,mktcp,ticker_norm,event_type,event_rank,has_event,n_announcements
0,2015-01-02,0001.hk,267.426000,131.70,131.00,132.30,130.20,2331309.0,3.067293e+05,NaN,NaN,NaN,NaN,0001.hk,none,99,False,0
1,2015-01-02,0002.hk,299.715800,67.25,67.00,67.45,66.40,1649366.0,1.106447e+05,NaN,NaN,NaN,NaN,0002.hk,none,99,False,0
2,2015-01-02,0003.hk,738.075600,17.76,17.78,17.86,17.38,4817803.0,8.518497e+04,NaN,NaN,NaN,NaN,0003.hk,none,99,False,0
3,2015-01-02,0004.hk,147.770000,57.25,56.50,57.50,56.10,3757908.0,2.143517e+05,NaN,NaN,NaN,NaN,0004.hk,none,99,False,0
4,2015-01-02,0005.hk,609.385700,74.00,73.60,74.10,73.50,8614012.0,6.353929e+05,NaN,NaN,NaN,NaN,0005.hk,none,99,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1301860,2025-09-30,9988.hk,186.405068,177.00,176.60,178.00,173.70,135460219.0,2.389275e+10,0.840444,1.109045,NaN,3.375856e+12,9988.hk,none,99,False,0
1301861,2025-09-30,9992.hk,273.973118,266.80,260.20,268.00,260.00,11330044.0,2.995829e+09,1.645465,0.332871,NaN,3.582972e+11,9992.hk,none,99,False,0
1301862,2025-09-30,9993.hk,2.986319,2.67,2.60,2.68,2.60,4506000.0,1.191838e+07,0.698359,NaN,NaN,1.080076e+10,9993.hk,none,99,False,0
1301863,2025-09-30,9997.hk,12.055474,8.89,8.86,8.89,8.86,980500.0,8.706575e+06,0.278707,2.939258,NaN,1.073907e+10,9997.hk,none,99,False,0


In [12]:
data["has_event"].nunique()

2

In [13]:
data["has_event"].describe()

count     1301865
unique          2
top         False
freq      1265137
Name: has_event, dtype: object

In [14]:
# =========================================================
# 5. 检查：先看键是否真的匹配上了
# =========================================================
print("data unique ticker_norm:", data["ticker_norm"].nunique())
print("rep2 unique ticker_norm:", rep2["ticker_norm"].nunique())
print("overlap ticker count:", len(set(data["ticker_norm"].dropna()) & set(rep2["ticker_norm"].dropna())))

print("daily_event shape:", daily_event.shape)
print("matched rows in data:", (data["event_type"] != "none").sum())
print(data[["date", "ticker", "ticker_norm", "event_type", "event_rank", "has_event", "n_announcements"]].head())
print(data["event_type"].value_counts(dropna=False))

# =========================================================
# 6. 如果你想专门检查哪些公告键没有对上
# =========================================================
unmatched_rep2 = daily_event.merge(
    data[["ticker_norm", "date"]].drop_duplicates(),
    on=["ticker_norm", "date"],
    how="left",
    indicator=True
)

print("rep2 daily events not matched into data:", (unmatched_rep2["_merge"] == "left_only").sum())
print(
    unmatched_rep2.loc[unmatched_rep2["_merge"] == "left_only", ["ticker_norm", "date"]]
    .head(20)
)

data unique ticker_norm: 994
rep2 unique ticker_norm: 855
overlap ticker count: 855
daily_event shape: (33096, 6)
matched rows in data: 36728
        date   ticker ticker_norm event_type  event_rank  has_event  n_announcements
0 2015-01-02  0001.hk     0001.hk       none          99      False                0
1 2015-01-02  0002.hk     0002.hk       none          99      False                0
2 2015-01-02  0003.hk     0003.hk       none          99      False                0
3 2015-01-02  0004.hk     0004.hk       none          99      False                0
4 2015-01-02  0005.hk     0005.hk       none          99      False                0
event_type
none                     1265137
execution                  35531
irrelevant_repurchase        707
completion                   187
mandate                      163
corporate_action              80
nextday                       32
cancellation                  28
Name: count, dtype: int64
rep2 daily events not matched into data: 0
Empt

In [15]:
data.to_csv(r"E:\26Spring\Final\code\data_with_event_type.csv", index=False, encoding="utf-8-sig")


In [16]:
# =========================================================
# 统一港股代码格式：
# 00001.XHKG / 1.HK / 0001.hk / 00001.hk  -> 0001.hk
# =========================================================
def normalize_hk_ticker(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip().lower()
    m = re.search(r"(\d+)", s)
    if not m:
        return pd.NA
    num = int(m.group(1))
    return f"{num:04d}.hk"

# =========================================================
# 0. 基础格式统一
# =========================================================
data_bool = data.copy()
rep2_bool = rep2.copy()

data_bool["date"] = pd.to_datetime(data_bool["date"], errors="coerce").dt.normalize()
rep2_bool["info_date"] = pd.to_datetime(rep2_bool["info_date"], errors="coerce").dt.normalize()

data_bool["ticker_norm"] = data_bool["ticker"].map(normalize_hk_ticker)
rep2_bool["ticker_norm"] = rep2_bool["order_book_id"].map(normalize_hk_ticker)

data_bool = data_bool.dropna(subset=["date", "ticker_norm"]).copy()
rep2_bool = rep2_bool.dropna(subset=["info_date", "ticker_norm"]).copy()

# =========================================================
# 1. 布尔标签列
# =========================================================
label_cols = [
    "hit_nextday",
    "hit_execution",
    "hit_completion",
    "hit_mandate",
    "hit_corporate_action",
    "hit_cancellation",
    "hit_irrelevant_repurchase"
]

missing_cols = [c for c in label_cols if c not in rep2_bool.columns]
print("missing label cols:", missing_cols)
if len(missing_cols) > 0:
    raise ValueError(f"rep2 缺少这些标签列: {missing_cols}")

rep2_bool[label_cols] = rep2_bool[label_cols].fillna(False).astype(bool)

# =========================================================
# 2. 聚合到 ticker × date
#    规则：同一股票同一天，只要有一条公告命中某标签 => 当天该标签=True
# =========================================================
daily_bool = (
    rep2_bool.groupby(["ticker_norm", "info_date"], as_index=False)[label_cols]
    .max()
    .rename(columns={"info_date": "date"})
)

# 附加几个辅助列（建议保留）
daily_bool["has_any_event"] = daily_bool[label_cols].any(axis=1)
daily_bool["n_event_types"] = daily_bool[label_cols].sum(axis=1).astype(int)

# 当天相关公告数（标题筛出来后的公告数）
daily_count = (
    rep2_bool.groupby(["ticker_norm", "info_date"], as_index=False)
    .size()
    .rename(columns={"info_date": "date", "size": "n_repurchase_announcements"})
)

daily_bool = daily_bool.merge(
    daily_count,
    on=["ticker_norm", "date"],
    how="left"
)

daily_bool["n_repurchase_announcements"] = (
    daily_bool["n_repurchase_announcements"].fillna(0).astype(int)
)

# =========================================================
# 3. merge 回 data
# =========================================================
cols_to_drop = label_cols + ["has_any_event", "n_event_types", "n_repurchase_announcements"]
data_bool = data_bool.drop(columns=[c for c in cols_to_drop if c in data_bool.columns], errors="ignore")

data_bool = data_bool.merge(
    daily_bool,
    on=["ticker_norm", "date"],
    how="left",
    validate="many_to_one"
)

# 填充缺失
for c in label_cols:
    data_bool[c] = data_bool[c].fillna(False).astype(bool)

data_bool["has_any_event"] = data_bool["has_any_event"].fillna(False).astype(bool)
data_bool["n_event_types"] = data_bool["n_event_types"].fillna(0).astype(int)
data_bool["n_repurchase_announcements"] = data_bool["n_repurchase_announcements"].fillna(0).astype(int)

# =========================================================
# 4. 检查
# =========================================================
print(data_bool.shape)
print(data_bool[label_cols + ["has_any_event", "n_event_types", "n_repurchase_announcements"]].head())

print("\n各标签命中行数：")
print(data_bool[label_cols].sum())

print("\n是否有任意事件：")
print(data_bool["has_any_event"].value_counts(dropna=False))

print("\n事件类型个数分布：")
print(data_bool["n_event_types"].value_counts().sort_index())

missing label cols: []


C:\Users\SHEN\AppData\Local\Temp\ipykernel_21284\1318170357.py:96: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data_bool[c] = data_bool[c].fillna(False).astype(bool)
C:\Users\SHEN\AppData\Local\Temp\ipykernel_21284\1318170357.py:96: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data_bool[c] = data_bool[c].fillna(False).astype(bool)
C:\Users\SHEN\AppData\Local\Temp\ipykernel_21284\1318170357.py:96: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=F

(1301865, 28)
   hit_nextday  hit_execution  hit_completion  hit_mandate  hit_corporate_action  hit_cancellation  hit_irrelevant_repurchase  has_any_event  n_event_types  n_repurchase_announcements
0        False          False           False        False                 False             False                      False          False              0                           0
1        False          False           False        False                 False             False                      False          False              0                           0
2        False          False           False        False                 False             False                      False          False              0                           0
3        False          False           False        False                 False             False                      False          False              0                           0
4        False          False           False        False        

In [17]:
data_bool.to_csv(
    r"E:\26Spring\Final\code\data_with_event_bools.csv",
    index=False,
    encoding="utf-8-sig"
)